In [1]:
import numpy as np
import os
import nibabel as nib

def fancier_print(msg):
    print(75 * "-")
    print(msg)
    print(75 * "-")
    return

# Define arguments from the command line
path_data = "/home/cherukara/Documents/SimProjects/Data_HistoMouse/kidney_spleens"
name_data = "dwi_denoise_shortsep"
INPUT_PROTOCOL_NAME = "KIDNEY_SHORTSEP_EXVIVO"
BVAL_THRESHOLD = 20
VASC_THRESHOLD = 300

# Load Reference Protocol
signal_arr = np.load(f"all_signals_all_substrates.npy")
ref_bvals = np.loadtxt(f"protocols/reference_protocol/ref.bval")
ref_delta = np.loadtxt(f"protocols/reference_protocol/ref.gdur")
ref_DELTA = np.loadtxt(f"protocols/reference_protocol/ref.gsep")

# Load Target Protocol
target_bvals = np.loadtxt(f"{path_data}/{name_data}.bval")
target_delta = np.loadtxt(f"{path_data}/{name_data}.gdur")
target_DELTA = np.loadtxt(f"{path_data}/{name_data}.gsep")

# Make protocol directory
os.makedirs(os.path.dirname(f"protocols/{INPUT_PROTOCOL_NAME}"), exist_ok=True)


This is the bash command that calls what we want to do:
```
python get_closest_scheme.py \
    --protocol-name KIDNEY_SHORTSEP_EXVIVO \
    --ref-signal all_signals_all_substrates.npy \
    --dwi <blah>/kidney_spleens/dwi_denoise_shortsep.nii \
    --bval <blah>/kidney_spleens/dwi_denoise_shortsep.bval \
    --gdur <blah>/kidney_spleens/dwi_denoise_shortsep.gdur \
    --gsep <blah>/kidney_spleens/dwi_denoise_shortsep.gsep \
    --bval-threshold 20 \
    --vasc-threshold 300 \
    --noise <blah>/kidney_spleens/dwi_noise.nii \
    --show-plot
```

In [2]:
# Load DWI
dwi = nib.load(f"{path_data}/{name_data}.nii")
data = dwi.get_fdata()
affine = dwi.affine
header = dwi.header

In [3]:
# Check for invalid b-values supplied
max_ref_bval = np.max(ref_bvals)
if np.any(target_bvals > max_ref_bval):
    print("Warning!")
    print(f"Max bvalue in the target scheme is {np.max(target_bvals)} s/mm2")
    print(f"Max bvalue synthesized signals are at {max_ref_bval} s/mm2 ")
    print(f"Volumes with bvalues higher than that are removed")
    too_high_locs = np.where(target_bvals > max_ref_bval)[0]
    data = np.delete(data, too_high_locs, axis=3)
    target_bvals = np.delete(target_bvals, too_high_locs)
    target_delta = np.delete(target_delta, too_high_locs)
    target_DELTA = np.delete(target_DELTA, too_high_locs)

if np.any((target_bvals < VASC_THRESHOLD) & (target_bvals > int(BVAL_THRESHOLD))):
    print()
    print("Warning!")
    print(f"bvalue under {VASC_THRESHOLD} s/mm2 in the target scheme")
    print(f"Min bvalue is 300 s/mm2 so that all effects of vascular flow are removed")
    print(f"Volumes with bvalues lower than that and higher than the bval threshold of {BVAL_THRESHOLD} are removed")
    too_low_locs = np.where((target_bvals < VASC_THRESHOLD) & (target_bvals > int(BVAL_THRESHOLD)))[0]
    data = np.delete(data, too_low_locs, axis=3)
    target_bvals = np.delete(target_bvals, too_low_locs)
    target_delta = np.delete(target_delta, too_low_locs)
    target_DELTA = np.delete(target_DELTA, too_low_locs)

In [4]:
# Level 1: Filter by closest b
if np.any(target_bvals > np.max(ref_bvals)):
    print("Warning!")
    print(f"Max bvalue in the target scheme is {np.max(target_bvals)}")
    print(f"Max bvalue for the synthesized signals is {np.max(ref_bvals)}")
b_min_inds = []
for b in target_bvals:
    diff1 = np.abs(ref_bvals - b).argmin()
    b_min_inds.append(diff1)
res_bvals = ref_bvals[b_min_inds]

In [5]:
# Level 2
delta1_min_inds = []
for d1 in target_delta:
    diff2 = np.abs(ref_delta - d1).argmin()
    delta1_min_inds.append(diff2)
res_delta = ref_delta[delta1_min_inds]

In [6]:
# Level 3
DELTA_min_inds = []
for D in target_DELTA:
    diff4 = np.abs(ref_DELTA - D).argmin()
    DELTA_min_inds.append(diff4)
res_DELTA = ref_DELTA[DELTA_min_inds]


In [7]:
# Print the scheme
print()
fancier_print("Resulting scheme")
print(res_bvals)
print(res_delta)
print(res_DELTA)
print()
fancier_print("Target scheme")
print(target_bvals)
print(target_delta)
print(target_DELTA)


---------------------------------------------------------------------------
Resulting scheme
---------------------------------------------------------------------------
[   0. 4600.  500. 2100.]
[11. 11. 11. 11.]
[16. 16. 16. 16.]

---------------------------------------------------------------------------
Target scheme
---------------------------------------------------------------------------
[   7.94 4628.46  520.07 2063.  ]
[12. 12. 12. 12.]
[16.5 16.5 16.5 16.5]


In [8]:
# Save the scheme
res_bvals = res_bvals.reshape(1,-1)
res_delta = res_delta.reshape(1,-1)
res_DELTA = res_DELTA.reshape(1,-1)
np.savetxt(f"protocols/{INPUT_PROTOCOL_NAME}/closest.bval", res_bvals, fmt="%.2f", delimiter=" ")
np.savetxt(f"protocols/{INPUT_PROTOCOL_NAME}/closest.gdur", res_delta, fmt="%.2f", delimiter=" ")
np.savetxt(f"protocols/{INPUT_PROTOCOL_NAME}/closest.gsep", res_DELTA, fmt="%.2f", delimiter=" ")

scheme = np.concatenate((res_bvals, res_delta, res_DELTA))
np.savetxt(f"protocols/{INPUT_PROTOCOL_NAME}/closest.scheme", scheme, fmt="%.2f", delimiter=" ")

print()
print(f"Saved closest protocol at: protocols/{INPUT_PROTOCOL_NAME}")


Saved closest protocol at: protocols/KIDNEY_SHORTSEP_EXVIVO


In [ ]:
# NIFTI processing

# Load the noise
noise = nib.load(f"{path_data}/dwi_noise.nii")
noise_data = noise.get_fdata()
noise_affine = noise.affine
noise_header = noise.header

print()
print(f"Normalizing the scan and noise, assuming that any volumes with b < {BVAL_THRESHOLD} s/mm2 are b = 0")

# Normalize by b=0
b0_vols = np.where(target_bvals <= int(BVAL_THRESHOLD))[0]
mean_b0 = np.mean(data[..., b0_vols], axis=-1)
new_data = data/mean_b0[..., np.newaxis]
new_noise_data = noise_data/mean_b0

# Save result
output_img = nib.Nifti1Image(new_data, affine, header)
nib.save(output_img, f"protocols/{INPUT_PROTOCOL_NAME}/dwi_normalized.nii")

output_noise = nib.Nifti1Image(new_noise_data, noise_affine, noise_header)
nib.save(output_noise, f"protocols/{INPUT_PROTOCOL_NAME}/dwi_noise_normalized.nii")



Normalizing the scan and noise, assuming that any volumes with b < 20 s/mm2 are b = 0
1


In [15]:
# Grab signal subset

# Find indices that match ALL four parameters
b_min_inds_arr = np.array(b_min_inds)
delta1_min_inds_arr = np.array(delta1_min_inds)
DELTA_min_inds_arr = np.array(DELTA_min_inds)

# Find the reference indices that match our final closest scheme
final_inds = []
for i in range(res_bvals.size):
    # Find where ALL parameters match in the reference arrays
    if res_bvals[:, i] != 0:
        candidates = np.where(
            (ref_bvals == res_bvals[:, i]) &
            (ref_delta == res_delta[:, i]) &
            (ref_DELTA == res_DELTA[:, i])
        )[0]
        # print(candidates)
        if len(candidates) > 0:
            final_inds.append(candidates[0])
        else:
            raise ValueError(f"No matching entry found for volume {i}")
    else:
        # when we have a b = 0 volume S = 1
        final_inds.append(0)

# Extract signal subset
signal_subset = signal_arr[:, final_inds]
np.save(f"protocols/{INPUT_PROTOCOL_NAME}/signal_arr_subset.npy", signal_subset)